In [1]:
# Imports
import os
import joblib
import random
import pickle
import numpy as np
import pandas as pd

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torch, random, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from statistics import mean, stdev
from sklearn.metrics import roc_auc_score, precision_score, recall_score, confusion_matrix, f1_score
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import RepeatedStratifiedKFold
from itertools import product
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score,
    average_precision_score, confusion_matrix
)
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from statistics import mean, stdev

# Ray Tune imports
import ray
from ray import tune, train
from ray.tune import CLIReporter
from ray.tune.schedulers import ASHAScheduler
import tempfile

import config
from preprocessing_utils import *
from model_utils import *

# Set seeds for reproducibility
set_random_seed(config.SEED, deterministic=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
def to_loader(c, m, mu, y, shuffle=False):
    
    def check_numeric(df, name):
        if isinstance(df, np.ndarray):
            df = pd.DataFrame(df)
        non_numeric_cols = []
        for col in df.columns:
            if not pd.api.types.is_numeric_dtype(df[col]):
                non_numeric_cols.append(col)
        if non_numeric_cols:
            print(f"WARNING: {name} has non-numeric columns: {non_numeric_cols}")
            # Print the first few rows of problematic columns
            print(df[non_numeric_cols].head())
        return df.to_numpy(dtype=np.float32)
    
    c = check_numeric(c, "Clinical")
    m = check_numeric(m, "mRNA")
    mu = check_numeric(mu, "Mutation")

    if isinstance(y, (pd.DataFrame, pd.Series)):
        y = y.to_numpy(dtype=np.float32).reshape(-1, 1)
    else:
        y = np.array(y, dtype=np.float32).reshape(-1, 1)
    y = y.squeeze() # converts from [x, 1] to [x] shape

    ds = TensorDataset(
        torch.tensor(c),
        torch.tensor(m),
        torch.tensor(mu),
        torch.tensor(y)
    )
    return DataLoader(ds, batch_size=config.BATCH_SIZE, shuffle=shuffle)


In [6]:
X_train = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "X_train.joblib"))
y_train = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "y_train.joblib"))
X_val = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "X_val.joblib"))
y_val = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "y_test.joblib"))
X_test = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "X_test.joblib"))
y_test = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "y_val.joblib"))
clinical_cols = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "clinical_cols.joblib"))
mrna_cols = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "mrna_cols.joblib"))
mutation_cols = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "mutation_cols.joblib"))

# Split modalities
clinical_train = X_train[clinical_cols]
mrna_train = X_train[mrna_cols]
mutation_train = X_train[mutation_cols]

clinical_val = X_val[clinical_cols]
mrna_val = X_val[mrna_cols]
mutation_val = X_val[mutation_cols]

clinical_test = X_test[clinical_cols]
mrna_test = X_test[mrna_cols]
mutation_test = X_test[mutation_cols]

In [7]:
def train_with_raytune(config_params):
    """
    Ray Tune trainable function for hyperparameter optimization.
    No feature selection - uses all features.
    Data is retrieved from Ray's object store to avoid serialization issues.
    """
    # Import all dependencies inside the function so Ray workers can access them
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    import numpy as np
    import pandas as pd
    import random
    import ray
    from sklearn.metrics import roc_auc_score, f1_score, average_precision_score
    import config
    from model_utils import GeneSelector, ModalityEncoder
    from preprocessing_utils import ClinicalPreprocessorWrapper, MrnaPreprocessorWrapper, MutationPreprocessorWrapper
    
    # Define MultimodalNet class inside trainable function
    class MultimodalNet(nn.Module):
        def __init__(self,
                     clin_dim, mrna_dim, mut_dim,
                     clin_hidden=[64, 32],
                     mrna_hidden=[64, 32],
                     mut_hidden=[64, 32],
                     clin_dropout=[0, 0],
                     mrna_dropout=[0, 0],
                     mut_dropout=[0, 0],
                     activation='relu',
                     fusion_hidden=64,
                     fusion_dropout=0,
                     use_gene_sel=True,
                    ):
            super().__init__()
            
            self.use_gene_sel = use_gene_sel
            if use_gene_sel:
                self.gene_sel_mrna = GeneSelector(mrna_dim)
                self.gene_sel_mut = GeneSelector(mut_dim)

            self.enc_clin = ModalityEncoder(clin_dim, clin_hidden, clin_dropout, activation)
            self.enc_mrna = ModalityEncoder(mrna_dim, mrna_hidden, mrna_dropout, activation)
            self.enc_mut  = ModalityEncoder(mut_dim, mut_hidden, mut_dropout, activation)

            total_dim = clin_hidden[-1] + mrna_hidden[-1] + mut_hidden[-1]
            self.fusion_fc = nn.Sequential(
                nn.Linear(total_dim, fusion_hidden),
                nn.ReLU(),
                nn.Dropout(fusion_dropout),
                nn.Linear(fusion_hidden, 1)
            )
            
        def forward(self, clin, mrna, mut):
            if self.use_gene_sel:
                mrna = self.gene_sel_mrna(mrna)
                mut = self.gene_sel_mut(mut)

            clin_emb = self.enc_clin(clin)
            mrna_emb = self.enc_mrna(mrna)
            mut_emb  = self.enc_mut(mut)
            
            fused = torch.cat([clin_emb, mrna_emb, mut_emb], dim=1)
            output = self.fusion_fc(fused)
            return output.squeeze()
    
    # Define to_loader function inside trainable
    def to_loader(c, m, mu, y, shuffle=False):
        def check_numeric(df, name):
            if isinstance(df, np.ndarray):
                df = pd.DataFrame(df)
            non_numeric_cols = []
            for col in df.columns:
                if not pd.api.types.is_numeric_dtype(df[col]):
                    non_numeric_cols.append(col)
            if non_numeric_cols:
                print(f"WARNING: {name} has non-numeric columns: {non_numeric_cols}")
                print(df[non_numeric_cols].head())
            return df.to_numpy(dtype=np.float32)
        
        c = check_numeric(c, "Clinical")
        m = check_numeric(m, "mRNA")
        mu = check_numeric(mu, "Mutation")

        if isinstance(y, (pd.DataFrame, pd.Series)):
            y = y.to_numpy(dtype=np.float32).reshape(-1, 1)
        else:
            y = np.array(y, dtype=np.float32).reshape(-1, 1)
        y = y.squeeze()

        ds = TensorDataset(
            torch.tensor(c),
            torch.tensor(m),
            torch.tensor(mu),
            torch.tensor(y)
        )
        return DataLoader(ds, batch_size=config.BATCH_SIZE, shuffle=shuffle)
    
    # Retrieve data from Ray's object store
    clin_train = ray.get(config_params["clinical_train_ref"])
    mrna_train = ray.get(config_params["mrna_train_ref"])
    mut_train = ray.get(config_params["mutation_train_ref"])
    y_train = ray.get(config_params["y_train_ref"])
    
    clin_val = ray.get(config_params["clinical_val_ref"])
    mrna_val = ray.get(config_params["mrna_val_ref"])
    mut_val = ray.get(config_params["mutation_val_ref"])
    y_val = ray.get(config_params["y_val_ref"])
    
    # Set seed for reproducibility
    seed = config_params["seed"] # FIXME: remove this, seed is already set, or use premade function
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    
    # ===== PREPROCESSING =====
    # Create preprocessors
    clinical_prep = ClinicalPreprocessorWrapper(
        cols_to_remove=config.CLINICAL_COLS_TO_REMOVE,
        categorical_cols=config.CATEGORICAL_COLS,
        max_null_frac=config.CLINICAL_MAX_NULL_FRAC,
        uniform_thresh=config.CLINICAL_UNIFORM_THRESH,
    )
    mrna_prep = MrnaPreprocessorWrapper(
        max_null_frac=config.MAX_NULL_FRAC,
        uniform_thresh=config.UNIFORM_THRESHOLD,
        random_state=config.SEED,
    )
    mutation_prep = MutationPreprocessorWrapper(
        max_mutation_count=config_params.get("max_mutation_count", 10),
        uniform_thresh=config_params.get("mutation_uniform_thresh", 0.95),
    )
    
    # Fit preprocessors on training data
    clinical_prep.fit(clin_train)
    mrna_prep.fit(mrna_train, y_train)
    mutation_prep.fit(mut_train)
    
    # Transform training and validation data
    clin_train = clinical_prep.transform(clin_train)
    clin_val = clinical_prep.transform(clin_val)
    
    mrna_train = mrna_prep.transform(mrna_train)
    mrna_val = mrna_prep.transform(mrna_val)
    
    mut_train = mutation_prep.transform(mut_train)
    mut_val = mutation_prep.transform(mut_val)
    
    # Device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Create data loaders with preprocessed data
    train_loader = to_loader(clin_train, mrna_train, mut_train, y_train, shuffle=True)
    val_loader = to_loader(clin_val, mrna_val, mut_val, y_val)
    
    # Get dimensions after preprocessing
    clin_dim = clin_train.shape[1]
    mrna_dim = mrna_train.shape[1]
    mut_dim = mut_train.shape[1]
    
    # Create model with hyperparameters from config_params
    # NOTE: use_gene_sel=False means no feature selection in the model
    model = MultimodalNet(
        clin_dim=clin_dim,
        mrna_dim=mrna_dim,
        mut_dim=mut_dim,
        clin_hidden=config_params["clin_hidden"],
        mrna_hidden=config_params["mrna_hidden"],
        mut_hidden=config_params["mut_hidden"],
        clin_dropout=config_params["clin_dropout"],
        mrna_dropout=config_params["mrna_dropout"],
        mut_dropout=config_params["mut_dropout"],
        activation=config_params.get("activation", "relu"),
        fusion_hidden=config_params["fusion_hidden"],
        fusion_dropout=config_params["fusion_dropout"],
        use_gene_sel=False  # No feature selection
    ).to(device)
    
    # Optimizer and loss
    optimizer = torch.optim.Adam(model.parameters(), lr=config_params["lr"])
    pos_weight_value = torch.tensor(
        (len(y_train) - y_train.sum()) / y_train.sum(), 
        dtype=torch.float32
    ).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_value)
    
    # Training loop
    best_val_auroc = 0
    patience_counter = 0
    patience = config_params["patience"]
    num_epochs = config_params["num_epochs"]
    
    for epoch in range(num_epochs):
        # Training
        model.train()
        for clin, mrna, mut, y in train_loader:
            clin = clin.to(device)
            mrna = mrna.to(device)
            mut = mut.to(device)
            y = y.to(device)
            
            optimizer.zero_grad()
            outputs = model(clin, mrna, mut)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()
        
        # Validation
        model.eval()
        all_probs, all_labels = [], []
        with torch.no_grad():
            for clin, mrna, mut, y in val_loader:
                clin = clin.to(device)
                mrna = mrna.to(device)
                mut = mut.to(device)
                y = y.to(device)
                
                probs = torch.sigmoid(model(clin, mrna, mut)).cpu().numpy()
                all_probs.append(probs)
                all_labels.append(y.cpu().numpy())
        
        all_probs = np.concatenate(all_probs)
        all_labels = np.concatenate(all_labels)
        
        # Calculate metrics
        val_auroc = roc_auc_score(all_labels, all_probs)
        val_auprc = average_precision_score(all_labels, all_probs)
        
        # Find optimal threshold for F1
        thresholds = np.linspace(0.001, 0.9, 200)
        best_f1 = 0
        for t in thresholds:
            preds = (all_probs >= t).astype(float)
            f1 = f1_score(all_labels, preds, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
        
        # Report metrics to Ray Tune
        from ray import train as ray_train
        tune.report({
            "auroc": val_auroc,
            "auprc": val_auprc,
            "f1": best_f1,
            "epoch": epoch
        })
        
        # Early stopping
        if val_auroc > best_val_auroc:
            best_val_auroc = val_auroc
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

In [ ]:
# Initialize Ray (if not already initialized)
if not ray.is_initialized():
    ray.init(ignore_reinit_error=True)

# Put data in Ray's object store to avoid serialization issues
clinical_train_ref = ray.put(clinical_train)
mrna_train_ref = ray.put(mrna_train)
mutation_train_ref = ray.put(mutation_train)
y_train_ref = ray.put(y_train)

clinical_val_ref = ray.put(clinical_val)
mrna_val_ref = ray.put(mrna_val)
mutation_val_ref = ray.put(mutation_val)
y_val_ref = ray.put(y_val)

print("Data loaded into Ray object store")

# Define hyperparameter search space for Ray Tune
search_space = {
    # Data references (fixed for all trials)
    "clinical_train_ref": clinical_train_ref,
    "mrna_train_ref": mrna_train_ref,
    "mutation_train_ref": mutation_train_ref,
    "y_train_ref": y_train_ref,
    "clinical_val_ref": clinical_val_ref,
    "mrna_val_ref": mrna_val_ref,
    "mutation_val_ref": mutation_val_ref,
    "y_val_ref": y_val_ref,
    
    # Fixed parameters
    "seed": config.SEED,
    "patience": config.PATIENCE,
    "num_epochs": config.NUM_EPOCHS,
    
    # Hidden layer sizes for each modality
    "clin_hidden": tune.choice([
        [64],
        [128],
    ]),
    "mrna_hidden": tune.choice([
        [64],
        [128],
        [32],
    ]),
    "mut_hidden": tune.choice([
        [32],
        [64],
        [16],
    ]),
    
    # Dropout rates for each modality
    "clin_dropout": tune.choice([
        [0.0],
        [0.2],
        [0.3],
    ]),
    "mrna_dropout": tune.choice([
        [0.0],
        [0.2],
        [0.3],
    ]),
    "mut_dropout": tune.choice([
        [0.0],
        [0.2],
        [0.3],
    ]),
    
    # Fusion layer parameters
    "fusion_hidden": tune.choice([64, 128, 256]),
    "fusion_dropout": tune.uniform(0.0, 0.5),
    
    # Learning rate
    "lr": tune.loguniform(1e-5, 1e-2),
    
    # Activation function

    "activation": tune.choice(["leaky_relu", "relu", "gelu", "silu"]),
}

In [ ]:
# Configure and run Ray Tune experiment
scheduler = ASHAScheduler(
    metric="f1",  # Optimize for F1 score
    mode="max",
    max_t=config.NUM_EPOCHS,
    grace_period=10,
    reduction_factor=2
)

reporter = CLIReporter(
    metric_columns=["auroc", "auprc", "f1", "epoch"],
    max_report_frequency=30
)

# Specify resources per trial
resources_per_trial = {
    "cpu": 1,
    "gpu": 0.25  # Each trial uses 1/4 of a GPU (allows 4 trials in parallel)
}

print("Starting Ray Tune hyperparameter search...")

# Run the hyperparameter search
result = tune.run(
    train_with_raytune,
    config=search_space,
    resources_per_trial=resources_per_trial,
    num_samples=50,  # Number of hyperparameter combinations to try
    scheduler=scheduler,
    progress_reporter=reporter,
    storage_path=tempfile.mkdtemp(),  # Temporary storage for checkpoints
    name="multimodal_hyperparam_search",
    verbose=1,
    raise_on_failed_trial=False  # Don't stop if a trial fails
)

# Get the best trial
best_trial = result.get_best_trial("f1", "max", "last")
print(f"\n{'='*80}")
print(f"Best trial config: {best_trial.config}")
print(f"Best trial final validation F1: {best_trial.last_result['f1']:.4f}")
print(f"Best trial final validation AUROC: {best_trial.last_result['auroc']:.4f}")
print(f"Best trial final validation AUPRC: {best_trial.last_result['auprc']:.4f}")
print(f"{'='*80}\n")

# Extract only the hyperparameters (not the data references)
best_hyperparams = {
    k: v for k, v in best_trial.config.items() 
    if not k.endswith("_ref") and k not in ["seed", "patience", "num_epochs"]
}

# Save the best hyperparameters
with open('best_hyperparams_raytune.pkl', 'wb') as f:
    pickle.dump(best_hyperparams, f)
    
print("Best hyperparameters saved to 'best_hyperparams_raytune.pkl'")

In [ ]:
def train_one_fold(train_loader, val_loader, model, optimizer, criterion, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    best_val_auroc, best_state, patience_counter = 0, None, 0

    for epoch in range(config.NUM_EPOCHS):
        model.train()
        for clin, mrna, mut, y in train_loader:
            clin, mrna, mut, y = clin.to(device), mrna.to(device), mut.to(device), y.to(device)
            optimizer.zero_grad()
            outputs = model(clin, mrna, mut)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()

        model.eval()
        all_probs, all_labels = [], []
        with torch.no_grad():
            for clin, mrna, mut, y in val_loader:
                clin, mrna, mut, y = clin.to(device), mrna.to(device), mut.to(device), y.to(device)
                probs = torch.sigmoid(model(clin, mrna, mut)).cpu().numpy()
                all_probs.append(probs)
                all_labels.append(y.cpu().numpy())

        all_probs = np.concatenate(all_probs)
        all_labels = np.concatenate(all_labels)
        val_auroc = roc_auc_score(all_labels, all_probs)

        if val_auroc > best_val_auroc:
            best_val_auroc = val_auroc
            best_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= config.PATIENCE:
                break

    return best_state

def run_kfold_gridsearch_with_preprocessing(
    clinical_df, mrna_df, mutation_df, labels,
    external_clinical_df, external_mrna_df, external_mutation_df, external_labels,
    param_grid,
    k=5,
    n_repeats=3,
    optimize_metric='f1',
):
    """
    Grid search with repeated stratified K-fold CV, fitting preprocessors and
    feature selectors (SelectFromModel) within each fold.
    """

    rskf = RepeatedStratifiedKFold(n_splits=k, n_repeats=n_repeats, random_state=config.SEED)
    indices = np.arange(len(labels))
    param_combinations = list(product(*param_grid.values()))
    best_hyperparams, best_score = None, -1
    results_summary = {}

    for params in param_combinations:
        param_dict = dict(zip(param_grid.keys(), params))
        print(f"\n===== Hyperparams: {param_dict} =====")

        fold_metrics = {'auroc': [], 'auprc': [], 'precision': [], 'recall': [], 'f1': []}
        external_metrics = {'auroc': [], 'auprc': [], 'precision': [], 'recall': [], 'f1': []}

        for fold, (train_idx, val_idx) in enumerate(rskf.split(indices, labels)):
            print(f"\n--- Fold {fold + 1}/{k} ---")

            clin_train, clin_val = clinical_df.iloc[train_idx], clinical_df.iloc[val_idx]
            mrna_train, mrna_val = mrna_df.iloc[train_idx], mrna_df.iloc[val_idx]
            mut_train, mut_val = mutation_df.iloc[train_idx], mutation_df.iloc[val_idx]
            y_train, y_val = labels.iloc[train_idx], labels.iloc[val_idx]

            clinical_prep = ClinicalPreprocessorWrapper(
                cols_to_remove=config.CLINICAL_COLS_TO_REMOVE,
                categorical_cols=config.CATEGORICAL_COLS,
                max_null_frac=config.CLINICAL_MAX_NULL_FRAC,
                uniform_thresh=config.CLINICAL_UNIFORM_THRESH,
            )
            mrna_prep = MrnaPreprocessorWrapper(
                max_null_frac=config.MAX_NULL_FRAC,
                uniform_thresh=config.UNIFORM_THRESHOLD,
                random_state=config.SEED,
            )
            mutation_prep = MutationPreprocessorWrapper(
                max_mutation_count=param_dict["max_mutation_count"],
                uniform_thresh=param_dict["mutation_uniform_thresh"],
            )

            clinical_prep.fit(clin_train)
            mrna_prep.fit(mrna_train, y_train)
            mutation_prep.fit(mut_train)

            clin_train = clinical_prep.transform(clin_train)
            clin_val = clinical_prep.transform(clin_val)
            clin_ext = clinical_prep.transform(external_clinical_df.copy())

            mrna_train = mrna_prep.transform(mrna_train)
            mrna_val = mrna_prep.transform(mrna_val)
            mrna_ext = mrna_prep.transform(external_mrna_df.copy())

            mut_train = mutation_prep.transform(mut_train)
            mut_val = mutation_prep.transform(mut_val)
            mut_ext = mutation_prep.transform(external_mutation_df.copy())

            # === SelectFromModel for mRNA ===
            mrna_model_cls = param_dict["mrna_model"]  # e.g., LogisticRegression, RandomForestClassifier
            mrna_model_params = param_dict.get("mrna_model_params", {})  # estimator hyperparameters
            
            sfm_mrna = SelectFromModel(
                estimator=mrna_model_cls(**mrna_model_params),
                threshold=param_dict.get("mrna_threshold", "median"),
                max_features=param_dict.get("mrna_max_features", None)
            )
            
            sfm_mrna.fit(mrna_train, y_train)
            mrna_train = sfm_mrna.transform(mrna_train)
            mrna_val   = sfm_mrna.transform(mrna_val)
            mrna_ext   = sfm_mrna.transform(mrna_ext)
            
            
            # === SelectFromModel for mutation ===
            mut_model_cls = param_dict["mut_model"]
            mut_model_params = param_dict.get("mut_model_params", {})
            
            sfm_mut = SelectFromModel(
                estimator=mut_model_cls(**mut_model_params),
                threshold=param_dict.get("mut_threshold", "median"),
                max_features=param_dict.get("mut_max_features", None)
            )
            
            sfm_mut.fit(mut_train, y_train)
            mut_train = sfm_mut.transform(mut_train)
            mut_val   = sfm_mut.transform(mut_val)
            mut_ext   = sfm_mut.transform(mut_ext)

            # --- Build datasets and dataloaders ---
            train_loader = to_loader(clin_train, mrna_train, mut_train, y_train, shuffle=True)
            val_loader = to_loader(clin_val, mrna_val, mut_val, y_val)
            ext_loader = to_loader(clin_ext, mrna_ext, mut_ext, external_labels)

            model = SimpleMultimodalNet(clin_train.shape[1], mrna_train.shape[1], mut_train.shape[1],
                                        param_dict["hidden_dim"], param_dict["dropout"], param_dict["lr"]).to(device)
            optimizer = torch.optim.Adam(model.parameters(), lr=param_dict.get('lr', 1e-3))
            pos_weight_value = torch.tensor((len(y_train) - y_train.sum()) / y_train.sum(), dtype=torch.float32).to(device)
            criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_value)

            best_state = train_one_fold(train_loader, val_loader, model, optimizer, criterion, config.SEED + fold)
            model.load_state_dict(best_state)

            # Threshold tuning, evaluation identical
            all_probs, all_labels = [], []
            with torch.no_grad():
                for clin, mrna, mut, y in val_loader:
                    clin, mrna, mut, y = clin.to(device), mrna.to(device), mut.to(device), y.to(device)
                    probs = torch.sigmoid(model(clin, mrna, mut)).cpu().numpy()
                    all_probs.append(probs)
                    all_labels.append(y.cpu().numpy())
            all_probs = np.concatenate(all_probs)
            all_labels = np.concatenate(all_labels)

            thresholds = np.linspace(0.001, 0.9, 200)
            best_t, best_metric = 0.5, -1
            for t in thresholds:
                preds = (all_probs >= t).astype(float)
                score = f1_score(all_labels, preds, zero_division=0)
                if score > best_metric:
                    best_metric, best_t = score, t

            val_metrics = evaluate_with_threshold(model, val_loader, best_t, device)
            ext_metrics = evaluate_with_threshold(model, ext_loader, best_t, device)

            print(f"Val metrics: {val_metrics}")
            print(f"Ext metrics: {ext_metrics}")

            for k_ in fold_metrics.keys():
                fold_metrics[k_].append(val_metrics[k_])
                external_metrics[k_].append(ext_metrics[k_])

        results_summary[str(param_dict)] = {
            "internal": {k: (mean(v), stdev(v)) for k, v in fold_metrics.items()},
            "external": {k: (mean(v), stdev(v)) for k, v in external_metrics.items()}
        }

        mean_f1 = results_summary[str(param_dict)]["internal"]["f1"][0]
        if mean_f1 > best_score:
            best_score = mean_f1
            best_hyperparams = param_dict

    print(f"\n=== Best hyperparameters: {best_hyperparams} (mean F1 = {best_score:.4f}) ===")
    return results_summary, best_hyperparams

In [ ]:
# Train final model with best hyperparameters on full training set
# and evaluate on test set

# Load best hyperparameters
with open('best_hyperparams_raytune.pkl', 'rb') as f:
    best_config = pickle.load(f)

print(f"Training final model with best hyperparameters:\n{best_config}\n")

clinical_prep = ClinicalPreprocessorWrapper(
    cols_to_remove=config.CLINICAL_COLS_TO_REMOVE,
    categorical_cols=config.CATEGORICAL_COLS,
    max_null_frac=config.CLINICAL_MAX_NULL_FRAC,
    uniform_thresh=config.CLINICAL_UNIFORM_THRESH,
)
mrna_prep = MrnaPreprocessorWrapper(
    max_null_frac=config.MAX_NULL_FRAC,
    uniform_thresh=config.UNIFORM_THRESHOLD,
    random_state=config.SEED,
)
mutation_prep = MutationPreprocessorWrapper(
    max_mutation_count=10,
    uniform_thresh=0.95,
)

# Fit preprocessors on training data
clinical_prep.fit(clinical_train)
mrna_prep.fit(mrna_train, y_train)
mutation_prep.fit(mutation_train)

# Transform training and validation data
clinical_train = clinical_prep.transform(clinical_train)
clinical_val = clinical_prep.transform(clinical_testval)

mrna_train = mrna_prep.transform(mrna_train)
mrna_val = mrna_prep.transform(mrna_testval)

mutation_train = mutation_prep.transform(mutation_train)
mutation_val = mutation_prep.transform(mutation_testval)


# Create data loaders
train_loader = to_loader(clinical_train, mrna_train, mutation_train, y_train, shuffle=True)
val_loader = to_loader(clinical_val, mrna_val, mutation_val, y_testval)

# Create final model
final_model = MultimodalNet(
    clin_dim=clinical_train.shape[1],
    mrna_dim=mrna_train.shape[1],
    mut_dim=mutation_train.shape[1],
    clin_hidden=best_config["clin_hidden"],
    mrna_hidden=best_config["mrna_hidden"],
    mut_hidden=best_config["mut_hidden"],
    clin_dropout=best_config["clin_dropout"],
    mrna_dropout=best_config["mrna_dropout"],
    mut_dropout=best_config["mut_dropout"],
    activation=best_config.get("activation", "relu"),
    fusion_hidden=best_config["fusion_hidden"],
    fusion_dropout=best_config["fusion_dropout"],
    use_gene_sel=False  # No feature selection
).to(device)

optimizer = torch.optim.Adam(final_model.parameters(), lr=best_config["lr"])
pos_weight_value = torch.tensor(
    (len(y_train) - y_train.sum()) / y_train.sum(), 
    dtype=torch.float32
).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_value)

# Train the final model
best_state = train_one_fold(train_loader, val_loader, final_model, optimizer, criterion, config.SEED)
final_model.load_state_dict(best_state)

# Find optimal threshold on validation set
final_model.eval()
all_probs, all_labels = [], []
with torch.no_grad():
    for clin, mrna, mut, y in val_loader:
        clin, mrna, mut, y = clin.to(device), mrna.to(device), mut.to(device), y.to(device)
        probs = torch.sigmoid(final_model(clin, mrna, mut)).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(y.cpu().numpy())

all_probs = np.concatenate(all_probs)
all_labels = np.concatenate(all_labels)

# thresholds = np.linspace(0.001, 0.9, 200)
# best_threshold, best_f1_val = 0.5, 0
# for t in thresholds:
#     preds = (all_probs >= t).astype(float)
#     f1 = f1_score(all_labels, preds, zero_division=0)
#     if f1 > best_f1_val:
#         best_f1_val = f1
#         best_threshold = t

# print(f"Optimal threshold: {best_threshold:.4f}")

# Evaluate on validation set
val_metrics = evaluate_with_threshold(final_model, val_loader, best_threshold, device)
print(f"\nValidation metrics:")
for k, v in val_metrics.items():
    print(f"  {k}: {v:.4f}")

# Save the final model
torch.save(final_model.state_dict(), 'final_model_raytune.pt')
print("\nFinal model saved to 'final_model_raytune.pt'")

In [ ]:
X_train = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "X_train.joblib"))
y_train = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "y_train.joblib"))
X_test = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "X_test.joblib"))
y_test = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "y_test.joblib"))
clinical_cols = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "clinical_cols.joblib"))
mrna_cols = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "mrna_cols.joblib"))
mutation_cols = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "mutation_cols.joblib"))

# Split modalities
clinical_train = X_train[clinical_cols]
mrna_train = X_train[mrna_cols]
mutation_train = X_train[mutation_cols]

clinical_val = X_val[clinical_cols]
mrna_val = X_val[mrna_cols]
mutation_val = X_val[mutation_cols]

clinical_test = X_test[clinical_cols]
mrna_test = X_test[mrna_cols]
mutation_test = X_test[mutation_cols]

clinical_testval = X_testval[clinical_cols]
mrna_testval = X_testval[mrna_cols]
mutation_testval = X_testval[mutation_cols]

print(mutation_train.shape, mutation_testval.shape)

In [ ]:
# Create SHAP beeswarm plot for final model
import shap
import matplotlib.pyplot as plt

print("Preparing data for SHAP analysis...")

# Preprocess test data the same way as training
clinical_prep = ClinicalPreprocessorWrapper(
    cols_to_remove=config.CLINICAL_COLS_TO_REMOVE,
    categorical_cols=config.CATEGORICAL_COLS,
    max_null_frac=config.CLINICAL_MAX_NULL_FRAC,
    uniform_thresh=config.CLINICAL_UNIFORM_THRESH,
)
mrna_prep = MrnaPreprocessorWrapper(
    max_null_frac=config.MAX_NULL_FRAC,
    uniform_thresh=config.UNIFORM_THRESHOLD,
    random_state=config.SEED,
)
mutation_prep = MutationPreprocessorWrapper(
    max_mutation_count=10,
    uniform_thresh=0.95,
)

# Fit on training data
clinical_prep.fit(clinical_train)
mrna_prep.fit(mrna_train, y_train)
mutation_prep.fit(mutation_train)

# Transform test data
clin_test_prep = clinical_prep.transform(clinical_testval.copy())
mrna_test_prep = mrna_prep.transform(mrna_testval.copy())
mut_test_prep = mutation_prep.transform(mutation_testval.copy())

non_numeric = clin_test_prep.select_dtypes(exclude=[np.number]).columns
print(list(non_numeric))

allowed = {
    "float64", "float32", "float16",
    "complex64", "complex128",
    "int64", "int32", "int16", "int8",
    "uint64", "uint32", "uint16", "uint8",
    "bool"
}

bad_cols = [col for col in clin_test_prep.columns if clin_test_prep[col].dtype.name not in allowed]
print("Columns with unsupported dtypes:", bad_cols)
bad_cols = [col for col in mrna_test_prep.columns if mrna_test_prep[col].dtype.name not in allowed]
print("Columns with unsupported dtypes:", bad_cols)
bad_cols = [col for col in mut_test_prep.columns if mut_test_prep[col].dtype.name not in allowed]
print("Columns with unsupported dtypes:", bad_cols)


# Concatenate all features for SHAP
# Create feature names
clin_feature_names = [f"clinical_{i}" for i in range(clin_test_prep.shape[1])]
mrna_feature_names = [f"mrna_{i}" for i in range(mrna_test_prep.shape[1])]
mut_feature_names = [f"mutation_{i}" for i in range(mut_test_prep.shape[1])]
all_feature_names = clin_feature_names + mrna_feature_names + mut_feature_names

# Concatenate features
X_test_combined = np.concatenate([
    clin_test_prep.to_numpy() if hasattr(clin_test_prep, 'to_numpy') else clin_test_prep,
    mrna_test_prep.to_numpy() if hasattr(mrna_test_prep, 'to_numpy') else mrna_test_prep,
    mut_test_prep.to_numpy() if hasattr(mut_test_prep, 'to_numpy') else mut_test_prep
], axis=1)

print(f"Combined feature matrix shape: {X_test_combined.shape}")

# Create a wrapper function for the model that takes concatenated input
def model_predict(X):
    """Wrapper function for SHAP that splits concatenated features back into modalities"""
    final_model.eval()
    
    # Split features back into modalities
    clin_dim = clin_test_prep.shape[1]
    mrna_dim = mrna_test_prep.shape[1]
    mut_dim = mut_test_prep.shape[1]
    
    clin = X[:, :clin_dim]
    mrna = X[:, clin_dim:clin_dim + mrna_dim]
    mut = X[:, clin_dim + mrna_dim:]
    
    # Convert to tensors
    clin_tensor = torch.tensor(clin, dtype=torch.float32).to(device)
    mrna_tensor = torch.tensor(mrna, dtype=torch.float32).to(device)
    mut_tensor = torch.tensor(mut, dtype=torch.float32).to(device)
    
    # Get predictions
    with torch.no_grad():
        outputs = final_model(clin_tensor, mrna_tensor, mut_tensor)
        probs = torch.sigmoid(outputs).cpu().numpy()
    
    return probs

print("Computing SHAP values...")
# Use a subset of test data for background (SHAP can be slow)
background = shap.sample(X_test_combined, min(100, len(X_test_combined)))
explainer = shap.KernelExplainer(model_predict, background)

# Compute SHAP values for test set (or subset if too large)
n_explain = min(200, len(X_test_combined))
shap_values = explainer.shap_values(X_test_combined[:n_explain])

print(f"SHAP values computed for {n_explain} samples")

# Create beeswarm plot
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test_combined[:n_explain], 
                  feature_names=all_feature_names,
                  plot_type="dot",
                  max_display=20,
                  show=False)
plt.title("SHAP Beeswarm Plot - Final Ray Tune Model", fontsize=14, pad=20)
plt.tight_layout()
plt.savefig('shap_beeswarm_raytune.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nPlot saved to 'shap_beeswarm_raytune.png'")

In [ ]:
# Create SHAP beeswarm plot for final model
# This cell is standalone - it loads the saved model from Ray Tune
import shap
import matplotlib.pyplot as plt
import pickle

print("Loading saved model and hyperparameters...")

# Load best hyperparameters
with open('best_hyperparams_raytune.pkl', 'rb') as f:
    best_config = pickle.load(f)

print(f"Loaded hyperparameters: {best_config}")

# Prepare data (same as training)
print("\nPreparing data for SHAP analysis...")

# Preprocess test data the same way as training
clinical_prep = ClinicalPreprocessorWrapper(
    cols_to_remove=config.CLINICAL_COLS_TO_REMOVE,
    categorical_cols=config.CATEGORICAL_COLS,
    max_null_frac=config.CLINICAL_MAX_NULL_FRAC,
    uniform_thresh=config.CLINICAL_UNIFORM_THRESH,
)
mrna_prep = MrnaPreprocessorWrapper(
    max_null_frac=config.MAX_NULL_FRAC,
    uniform_thresh=config.UNIFORM_THRESHOLD,
    random_state=config.SEED,
)
mutation_prep = MutationPreprocessorWrapper(
    max_mutation_count=10,
    uniform_thresh=0.95,
)

# Fit on training data
clinical_prep.fit(clinical_train)
mrna_prep.fit(mrna_train, y_train)
mutation_prep.fit(mutation_train)

# Transform test data
clin_test_prep = clinical_prep.transform(clinical_test.copy())
mrna_test_prep = mrna_prep.transform(mrna_test.copy())
mut_test_prep = mutation_prep.transform(mutation_test.copy())

# Convert to numpy arrays and ensure they are numeric
def to_numeric_array(data):
    """Convert data to numeric numpy array"""
    if isinstance(data, pd.DataFrame):
        return data.to_numpy(dtype=np.float32)
    elif isinstance(data, np.ndarray):
        return data.astype(np.float32)
    else:
        return np.array(data, dtype=np.float32)

clin_test_array = to_numeric_array(clin_test_prep)
mrna_test_array = to_numeric_array(mrna_test_prep)
mut_test_array = to_numeric_array(mut_test_prep)

# Get actual feature names from preprocessed DataFrames
if isinstance(clin_test_prep, pd.DataFrame):
    clin_feature_names = list(clin_test_prep.columns)
else:
    clin_feature_names = [f"clinical_{i}" for i in range(clin_test_array.shape[1])]

if isinstance(mrna_test_prep, pd.DataFrame):
    mrna_feature_names = list(mrna_test_prep.columns)
else:
    mrna_feature_names = [f"mrna_{i}" for i in range(mrna_test_array.shape[1])]

if isinstance(mut_test_prep, pd.DataFrame):
    mut_feature_names = list(mut_test_prep.columns)
else:
    mut_feature_names = [f"mutation_{i}" for i in range(mut_test_array.shape[1])]

all_feature_names = clin_feature_names + mrna_feature_names + mut_feature_names

print(f"Number of features: Clinical={len(clin_feature_names)}, mRNA={len(mrna_feature_names)}, Mutation={len(mut_feature_names)}")
print(f"Sample clinical features: {clin_feature_names[:5]}")
print(f"Sample mRNA features: {mrna_feature_names[:5]}")
print(f"Sample mutation features: {mut_feature_names[:5]}")

# Concatenate features
X_test_combined = np.concatenate([clin_test_array, mrna_test_array, mut_test_array], axis=1).astype(np.float32)

print(f"\nCombined feature matrix shape: {X_test_combined.shape}")
print(f"Data type: {X_test_combined.dtype}")

# Store dimensions for splitting
clin_dim = clin_test_array.shape[1]
mrna_dim = mrna_test_array.shape[1]
mut_dim = mut_test_array.shape[1]

# Recreate the model with saved hyperparameters
print("\nRecreating model architecture...")
loaded_model = MultimodalNet(
    clin_dim=clin_dim,
    mrna_dim=mrna_dim,
    mut_dim=mut_dim,
    clin_hidden=best_config["clin_hidden"],
    mrna_hidden=best_config["mrna_hidden"],
    mut_hidden=best_config["mut_hidden"],
    clin_dropout=best_config["clin_dropout"],
    mrna_dropout=best_config["mrna_dropout"],
    mut_dropout=best_config["mut_dropout"],
    activation=best_config.get("activation", "relu"),
    fusion_hidden=best_config["fusion_hidden"],
    fusion_dropout=best_config["fusion_dropout"],
    use_gene_sel=False  # No feature selection
).to(device)

# Load saved weights
loaded_model.load_state_dict(torch.load('final_model_raytune.pt', map_location=device))
loaded_model.eval()

print("Model loaded successfully!")

# Create a wrapper function for the model that takes concatenated input
def model_predict(X):
    """Wrapper function for SHAP that splits concatenated features back into modalities"""
    loaded_model.eval()
    
    # Ensure X is float32 numpy array and 2D
    if not isinstance(X, np.ndarray):
        X = np.array(X, dtype=np.float32)
    else:
        X = X.astype(np.float32)
    
    # Ensure 2D shape
    if len(X.shape) == 1:
        X = X.reshape(1, -1)
    
    # Split features back into modalities
    clin = X[:, :clin_dim]
    mrna = X[:, clin_dim:clin_dim + mrna_dim]
    mut = X[:, clin_dim + mrna_dim:]
    
    # Convert to tensors
    clin_tensor = torch.tensor(clin, dtype=torch.float32).to(device)
    mrna_tensor = torch.tensor(mrna, dtype=torch.float32).to(device)
    mut_tensor = torch.tensor(mut, dtype=torch.float32).to(device)
    
    # Get predictions
    with torch.no_grad():
        outputs = loaded_model(clin_tensor, mrna_tensor, mut_tensor)
        probs = torch.sigmoid(outputs).cpu().numpy()
    
    # Ensure output is 1D for single sample or 2D column vector for multiple samples
    if probs.ndim == 0:
        probs = np.array([probs])
    elif probs.ndim == 1:
        pass  # Keep as is
    
    return probs

print("\nComputing SHAP values (this may take several minutes)...")
# Use DeepExplainer instead of KernelExplainer (faster for neural networks)
# First create a small background dataset
n_background = min(50, len(X_test_combined))
background = X_test_combined[:n_background]

# Create a PyTorch wrapper for the model
class ModelWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        
    def forward(self, x):
        # Split concatenated input
        clin = x[:, :clin_dim]
        mrna = x[:, clin_dim:clin_dim + mrna_dim]
        mut = x[:, clin_dim + mrna_dim:]
        
        outputs = self.model(clin, mrna, mut)
        return torch.sigmoid(outputs).unsqueeze(1)

wrapped_model = ModelWrapper(loaded_model)
wrapped_model.eval()

# Use GradientExplainer (faster than KernelExplainer)
background_tensor = torch.tensor(background, dtype=torch.float32).to(device)
explainer = shap.GradientExplainer(wrapped_model, background_tensor)

# Compute SHAP values for test set (or subset if too large)
n_explain = min(100, len(X_test_combined))  # Reduced for speed
X_explain = torch.tensor(X_test_combined[:n_explain], dtype=torch.float32).to(device)
shap_values = explainer.shap_values(X_explain)

# Convert to numpy if needed
if isinstance(shap_values, torch.Tensor):
    shap_values = shap_values.cpu().numpy()
elif isinstance(shap_values, list):
    shap_values = shap_values[0].cpu().numpy() if isinstance(shap_values[0], torch.Tensor) else shap_values[0]

# Remove extra dimension if present
if shap_values.ndim == 3:
    shap_values = shap_values[:, :, 0]

print(f"SHAP values computed for {n_explain} samples")
print(f"SHAP values shape: {shap_values.shape}")

# Create beeswarm plot
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test_combined[:n_explain], 
                  feature_names=all_feature_names,
                  plot_type="dot",
                  max_display=30,
                  show=False)
plt.title("SHAP Beeswarm Plot - Final Ray Tune Model", fontsize=14, pad=20)
plt.tight_layout()
plt.savefig('shap_beeswarm_raytune.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nPlot saved to 'shap_beeswarm_raytune.png'")

In [7]:
from sklearn.inspection import permutation_importance

r = permutation_importance(final_model, X_val, y_val, n_repeats=10, random_state=42)
importances = r.importances_mean

# Check mean importance of expression features
expr_imps = importances[[i for i, c in enumerate(X_val.columns) if "expr_" in c]]
mut_imps  = importances[[i for i, c in enumerate(X_val.columns) if "mut_"  in c]]

print(expr_imps.mean(), mut_imps.mean())


NameError: name 'final_model' is not defined